# Три гипотезы о календарном прогнозе спроса

Проверяем небольшой вклад нелинейных сочетаний признаков и одно явное линейное
взаимодействие. Отклонение — это разница итогового спроса и среднего своей зоны
и сезона в обучающей истории. Все модели учат именно это отклонение.

Общий код берём из выбранного ноутбука, сохраняя подготовку данных, сезонный
уровень, поправку продаж и ограничения. Финальный расчёт test не запускаем;
этот эксперимент не записывает прогнозы для сдачи.

Используем согласованный для этой серии сценарный протокол: три окна,
утро открытия и утро O+27/O+7, единая выборка L≥28. Поздний раздел
«Режим финального дня» в PROTOCOL задаёт другой режим; в этой серии по прямому
поручению применяем сценарный режим исходного решения, чтобы сравнение было
сопоставимым. PROTOCOL остаётся без изменений.

Все три окна уже участвовали в выборе текущей модели. Независимой итоговой
оценки нет; заранее ограниченная серия не устраняет это ограничение.

In [1]:
from pathlib import Path
import nbformat
from catboost import CatBoostRegressor

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

source = nbformat.read(root / "notebooks/Avangard_ticket_demand.ipynb", as_version=4)
# Только подготовка и общие функции; нет ячейки сохранения итогового CSV.
common_cells = [1, 3, 5, 7, 9, 11, 13, 15, 17]
for number in common_cells:
    exec(compile(source.cells[number].source, f"Avangard_cell_{number}", "exec"))

Обучение: 85 матчей, 680 пар матч–зона.


scenario,Без продаж,С продажами,"С продажами, L 28–42"
window,,,
M035–M051,17,10,10
M052–M068,17,17,12
M069–M085,17,9,9


## 1. Заранее фиксируем варианты

А: сезонный уровень + 80% отклонения Ridge(alpha=10) + 20% отклонения CatBoost.
Б: сезонный уровень + Ridge(alpha=30), исходные признаки и категория zone×weekday.
Это одновременно взаимодействие и более сильный штраф; результат нельзя
приписать только взаимодействию.
В: 80% среднего зоны + 20% полного календарного прогноза CatBoost.

В А и В используется один и тот же CatBoost. Во всех трёх кандидатах после
смешивания календарных прогнозов применяется прежняя поправка ранних продаж.
В — смесь календарных прогнозов с общей поправкой, а не смесь полных итогов.
Baseline остаётся обычным средним зоны с общими границами, без модельной поправки.

CatBoost строит небольшие деревья: несколько последовательных вопросов о
признаках позволяют учитывать их сочетания. Глубина 3 ограничивает число
таких вопросов. Штраф 10 сдерживает случайные эффекты малой истории.
Число деревьев и веса не выбираются по проверочным ответам.

Для воспроизведения установите `catboost==1.2.8` дополнительно к
`requirements.txt` и выполните этот ноутбук сверху вниз. Версии запуска: Python 3.12.14; CatBoost 1.2.8; numpy 2.3.5; pandas 3.0.1; scikit-learn 1.7.2.

In [2]:
catboost_params = dict(
    iterations=300, depth=3, learning_rate=0.03, l2_leaf_reg=10,
    loss_function="RMSE", random_seed=42, use_best_model=False,
    thread_count=2, allow_writing_files=False, verbose=False,
)
interaction_alpha = 30
catboost_weight = 0.2
method_names = ["Baseline", "Текущая Ridge", "А: Ridge 80% + CB 20%",
                "Б: Ridge zone×weekday", "В: baseline 80% + CB 20%"]
print("CatBoost:", catboost_params)
print("Ridge взаимодействия: alpha=", interaction_alpha)

CatBoost: {'iterations': 300, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'loss_function': 'RMSE', 'random_seed': 42, 'use_best_model': False, 'thread_count': 2, 'allow_writing_files': False, 'verbose': False}
Ridge взаимодействия: alpha= 30


## 2. Учим календарные отклонения на доступной истории

Категорию zone×weekday можно представить как отдельную метку «зона 6,
суббота»: она позволяет линейной модели учитывать особую связь дня недели
с конкретной зоной. CatBoost получает исходные категории непосредственно.

Средние обучающих ответов, кодирование, масштабирование и обе модели
обучаются только на history. Истинные итоги query в функции не передаются.
CatBoost не получает eval_set и не останавливается по проверочному окну.

In [3]:
def interaction_features(frame):
    # Доступен на момент: расписание и зона известны до открытия продаж.
    features = model_features(frame)
    features["zone_weekday"] = features["zone"] + "_" + features["weekday"]
    return features


def alternative_calendars(history, query):
    # Доступен на момент: завершённая история и известное расписание query.
    centers = history.groupby(["zone", "season"])["tickets_total"].transform("mean")
    deviations = history["tickets_total"] - centers
    level = season_level(history, query)
    current = calendar_forecast(history, query)
    baseline = zone_mean(history, query)

    catboost = CatBoostRegressor(**catboost_params)
    catboost.fit(model_features(history), deviations, cat_features=categorical_features)
    catboost_deviation = catboost.predict(model_features(query))
    catboost_calendar = level + catboost_deviation

    categories = categorical_features + ["zone_weekday"]
    preprocessing = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categories),
        ("number", StandardScaler(), numeric_features),
    ])
    ridge = make_pipeline(preprocessing, Ridge(alpha=interaction_alpha, random_state=42))
    ridge.fit(interaction_features(history), deviations)
    interaction = level + ridge.predict(interaction_features(query))

    return {
        method_names[0]: baseline,
        method_names[1]: current,
        method_names[2]: (1 - catboost_weight) * current + catboost_weight * catboost_calendar,
        method_names[3]: interaction,
        method_names[4]: (1 - catboost_weight) * baseline + catboost_weight * catboost_calendar,
    }

## 3. Сохраняем общую поправку собственных продаж

Каждый кандидат получает одну и ту же поправку, рассчитанную на завершённых
аналогах при одинаковом числе дней до игры. Меняется только ожидаемый полный
итог до поправки: поэтому ожидаемые ранние продажи тоже соответствуют
новому календарному прогнозу. Baseline остаётся исходным ориентиром.

In [4]:
def add_sales_correction(history, query, initial):
    prediction = pd.Series(initial, index=query.index)
    for match_id, match_query in query.loc[query["has_sales"]].groupby("match_id"):
        cutoff = match_query["cutoff"].iloc[0]
        analogs = matched_history(history, match_id, cutoff)
        base = prediction.loc[match_query.index].to_numpy()
        prediction.loc[match_query.index] += sales_correction(analogs, match_query, base)
    return prediction.to_numpy()

## 4. Проверяем все методы на одинаковых примерах

Для каждого утра history содержит только матчи до начала окна и с датой
строго раньше cutoff. Все восемь зон сохраняются вместе. Проверочные матчи
не добавляются в историю даже после завершения внутри окна.

Продажи query берутся строго до cutoff; ответы присоединяются только после
получения прогноза. Для всех методов одинаковы округление floor(p+0.5),
минимум sold и максимум seats. Исторические итоговые абонементы исключены.
Фиксированная выборка содержит 984 прогноза на метод.

In [5]:
import time

started = time.perf_counter()
actual_totals = train.set_index(keys)["tickets_total"]
result_parts = []
group_count = examples.groupby(["window", "cutoff"]).ngroups
for step, ((window, cutoff), query) in enumerate(
    examples.groupby(["window", "cutoff"]), start=1
):
    first_id = query["window_start"].iloc[0]
    history = train.loc[(train["match_id"] < first_id) & (train["date"] < cutoff)]
    calendars = alternative_calendars(history, query)
    for name, initial in calendars.items():
        raw = initial if name == "Baseline" else add_sales_correction(history, query, initial)
        prediction = apply_bounds(raw, query["sold"], query["seats"])
        detail = query[
            keys + ["window", "scenario", "days_observed", "weight", "sales_length"]
        ].copy()
        detail["prediction"] = prediction
        # Ответ присоединяем только после обучения и прогноза.
        pairs = pd.MultiIndex.from_frame(query[keys])
        detail["tickets_total"] = actual_totals.reindex(pairs).to_numpy()
        detail["method"] = name
        detail["squared_error"] = (detail["prediction"] - detail["tickets_total"]) ** 2
        result_parts.append(detail)
    if step % 20 == 0 or step == group_count:
        print(f"Проверено отсечек: {step}/{group_count}")

validation = pd.concat(result_parts, ignore_index=True)
print(f"Время сравнения пяти методов: {time.perf_counter() - started:.1f} с")

Проверено отсечек: 20/47


Проверено отсечек: 40/47


Проверено отсечек: 47/47
Время сравнения пяти методов: 304.1 с


## 5. Считаем ошибки по сценарию и окну

Сначала смешиваем квадраты ошибок 27/7 с весами 2/3 и 1/3 внутри пары
матч–зона. Затем усредняем пары. Ошибка окна R смешивает квадраты сценарных
ошибок с весами 11/17 и 6/17. S — среднее R с весами 0.25/0.25/0.5.
Показываем также диагностику похожих сроков продаж L=28–42.

In [6]:
validation["weighted_error"] = validation["weight"] * validation["squared_error"]
pair_errors = validation.groupby(
    ["method", "window", "scenario"] + keys
)["weighted_error"].sum().reset_index()
scenario_mse = pair_errors.groupby(
    ["method", "window", "scenario"]
)["weighted_error"].mean().unstack("scenario")
window_results = np.sqrt(scenario_mse)
window_results["R"] = np.sqrt(
    (11 / 17) * scenario_mse["Без продаж"] + (6 / 17) * scenario_mse["С продажами"]
)
similar = validation.loc[
    (validation["scenario"] == "С продажами") & validation["sales_length"].between(28, 42)
]
similar_pairs = similar.groupby(["method", "window"] + keys)["weighted_error"].sum()
window_results["С продажами, L 28–42"] = np.sqrt(
    similar_pairs.groupby(["method", "window"]).mean()
)
display(window_results.reindex(method_names, level="method").round(3))

window_weights = np.array([0.25, 0.25, 0.50])
score_rows = []
for name in method_names:
    group = window_results.loc[name].sort_index()
    score_rows.append({
        "Метод": name,
        "R1": group["R"].iloc[0], "R2": group["R"].iloc[1], "R3": group["R"].iloc[2],
        "S без продаж": window_weights @ group["Без продаж"].to_numpy(),
        "S с продажами": window_weights @ group["С продажами"].to_numpy(),
        "S": window_weights @ group["R"].to_numpy(),
    })
scores = pd.DataFrame(score_rows).set_index("Метод")
scores["Изменение S, %"] = 100 * (scores["S"] / scores.loc["Текущая Ridge", "S"] - 1)
display(scores.round(3))

scenario                            Без продаж  С продажами        R  \
method                   window                                        
Baseline                 M035–M051     155.995      151.394  154.386   
                         M052–M068     105.410      102.931  104.542   
                         M069–M085     157.123      132.333  148.846   
Текущая Ridge            M035–M051     156.285      136.536  149.613   
                         M052–M068      73.740       66.976   71.426   
                         M069–M085     150.871      127.162  142.952   
А: Ridge 80% + CB 20%    M035–M051     157.889      138.189  151.229   
                         M052–M068      72.308       65.717   70.052   
                         M069–M085     151.120      127.515  143.234   
Б: Ridge zone×weekday    M035–M051     154.890      136.006  148.499   
                         M052–M068      73.359       66.614   71.052   
                         M069–M085     151.782      128.164  143.890   
В: baseline 80% + CB 20% M035–M051     155.942      133.745  148.487   
                         M052–M068      96.051       87.385   93.085   
                         M069–M085     155.805      127.763  146.522   

scenario                            С продажами, L 28–42  
method                   window                           
Baseline                 M035–M051               151.394  
                         M052–M068                99.279  
                         M069–M085               132.333  
Текущая Ridge            M035–M051               136.536  
                         M052–M068                65.291  
                         M069–M085               127.162  
А: Ridge 80% + CB 20%    M035–M051               138.189  
                         M052–M068                64.736  
                         M069–M085               127.515  
Б: Ridge zone×weekday    M035–M051               136.006  
                         M052–M068                64.411  
                         M069–M085               128.164  
В: baseline 80% + CB 20% M035–M051               133.745  
                         M052–M068                84.200  
                         M069–M085               127.763

,R1,R2,R3,S без продаж,S с продажами,S,"Изменение S, %"
Метод,,,,,,,
Baseline,154.386,104.542,148.846,143.913,129.748,139.155,9.799
Текущая Ridge,149.613,71.426,142.952,132.941,114.459,126.736,0.000
А: Ridge 80% + CB 20%,151.229,70.052,143.234,133.109,114.734,126.937,0.159
Б: Ridge zone×weekday,148.499,71.052,143.890,132.953,114.737,126.833,0.076
В: baseline 80% + CB 20%,148.487,93.085,146.522,140.901,119.164,133.654,5.459


## 6. Решение по заранее согласованным порогам

Нужны одновременно: снижение S не менее 2%, улучшение хотя бы двух окон,
ухудшение третьего не больше 3% и ухудшение любого сценарного S не больше 3%.
Если критерии не выполнены, сохраняем текущую модель. Отрицательное изменение
S означает улучшение. Порог — практическое правило выбора, а не доказательство
статистической значимости.

In [7]:
current = scores.loc["Текущая Ridge"]
acceptance_rows = []
for name in method_names[2:]:
    row = scores.loc[name]
    improved_windows = sum(row[f"R{i}"] < current[f"R{i}"] for i in (1, 2, 3))
    conditions = [row["S"] <= 0.98 * current["S"], improved_windows >= 2,
                  row["R3"] <= 1.03 * current["R3"],
                  row["S без продаж"] <= 1.03 * current["S без продаж"],
                  row["S с продажами"] <= 1.03 * current["S с продажами"]]
    acceptance_rows.append({
        "Метод": name, "Выигрыш ≥2%": conditions[0], "Окон лучше": improved_windows,
        "Окно 3 допустимо": conditions[2], "Без продаж допустимо": conditions[3],
        "С продажами допустимо": conditions[4], "Принять": all(conditions),
    })
acceptance = pd.DataFrame(acceptance_rows).set_index("Метод")
display(acceptance)

# Контроль воспроизведения исходной оценки, не подбор параметров.
reference = np.array([139.154980934, 126.735841108])
if not np.allclose(scores.loc[method_names[:2], "S"], reference, rtol=0, atol=1e-8):
    raise ValueError("Исходные оценки не воспроизвелись: сравнение нужно проверить.")
print("Baseline и текущая модель воспроизведены.")

,Выигрыш ≥2%,Окон лучше,Окно 3 допустимо,Без продаж допустимо,С продажами допустимо,Принять
Метод,,,,,,
А: Ridge 80% + CB 20%,False,1,True,True,True,False
Б: Ridge zone×weekday,False,2,True,True,True,False
В: baseline 80% + CB 20%,False,1,True,False,False,False


Baseline и текущая модель воспроизведены.


## 7. Диагностика по зонам

Для каждой зоны считаем R с теми же сценарными весами. Таблица содержит
восемь зон; она помогает объяснить результат, но не задаёт новые параметры
для отдельных зон. Параметры серии после просмотра ошибок не меняем.

In [8]:
zone_mse = pair_errors.groupby(
    ["method", "zone", "window", "scenario"]
)["weighted_error"].mean().unstack("scenario")
zone_rmse = np.sqrt(
    (11 / 17) * zone_mse["Без продаж"] + (6 / 17) * zone_mse["С продажами"]
)
for window in window_results.index.get_level_values("window").unique():
    print(window)
    display(zone_rmse.xs(window, level="window").unstack("method")[method_names].round(1))

M035–M051


method,Baseline,Текущая Ridge,А: Ridge 80% + CB 20%,Б: Ridge zone×weekday,В: baseline 80% + CB 20%
zone,,,,,
1,49.5,73.2,70.6,67.3,56.1
2,62.0,63.7,64.9,56.7,57.6
3,70.2,82.7,83.5,76.2,69.5
4,126.0,139.3,148.4,147.1,140.3
5,148.2,141.1,139.7,138.0,136.1
6,336.1,311.5,313.0,309.6,319.4
7,156.2,145.3,146.9,147.4,144.9
8,65.4,73.0,75.8,67.6,61.9


M052–M068


method,Baseline,Текущая Ridge,А: Ridge 80% + CB 20%,Б: Ridge zone×weekday,В: baseline 80% + CB 20%
zone,,,,,
1,45.1,38.2,36.0,33.6,39.7
2,29.6,35.1,31.8,28.7,28.0
3,57.3,68.9,66.8,65.2,58.8
4,117.6,78.5,75.5,83.2,98.5
5,111.4,99.6,96.3,105.0,102.5
6,209.6,75.6,74.5,69.3,181.2
7,85.7,96.2,97.6,93.9,84.3
8,60.9,48.2,49.4,51.1,58.2


M069–M085


method,Baseline,Текущая Ridge,А: Ridge 80% + CB 20%,Б: Ridge zone×weekday,В: baseline 80% + CB 20%
zone,,,,,
1,31.5,44.5,41.4,34.3,30.6
2,55.1,43.3,41.1,39.3,50.4
3,74.4,69.4,69.0,63.2,70.3
4,137.9,148.7,150.5,152.7,139.5
5,118.6,113.2,114.1,112.2,118.4
6,272.9,273.7,273.4,278.1,271.2
7,232.5,206.2,207.4,207.1,225.4
8,77.9,49.0,50.2,52.7,74.2


## 8. Проверка на утечки и границы выводов

1. Расписание и zone×weekday известны до прогноза; комментарии доступности
есть в функциях. КХЛ и исторические итоговые абонементы не входят в признаки.
2. Центры обучающих ответов и сезонный уровень считаются только из history.
Центрирование ответа — преобразование обучающей цели, не признак query.
3. Продажи строго до cutoff; примеры и веса одинаковы у всех пяти методов.
Аналоги завершены к cutoff, собственные их срезы раньше даты игры.
4. Окна идут по времени, все зоны матча вместе; проверочные матчи не пополняют history.
5. Encoder, scaler, Ridge и CatBoost обучаются только на history. CatBoost
не видит проверочных ответов при обучении, eval_set или выборе числа деревьев.
6. Исторические границы sold/seats и округление общие. Test не рассчитываем;
финальный CSV остаётся без изменений. Историческая доступность seats —
согласованное допущение, доступность финальных абонементов не установлена.
7. Контроль исходных двух оценок выполняется программно. Все новые числа
описывают повторно использованные окна выбора; независимой итоговой оценки нет.


## 9. Вывод

Сохраняем текущую Ridge с сезонным уровнем и прежней поправкой продаж.
Ни один кандидат не выполнил заранее заданные условия принятия.

- А: S=126.937 против 126.736, ухудшение 0.159%; лучше только второе окно.
  Малый вклад нелинейной модели не дал общего выигрыша.
- Б: S=126.833, ухудшение 0.076%; первое и второе окна лучше, третье хуже.
  В последнем окне ошибка зоны 6 выросла с 273.7 до 278.1 билета.
  Взаимодействие и более сильный штраф оценивались вместе.
- В: S=133.654, ухудшение 5.459%; сценарные S ухудшились больше 3%.
  Большой вес среднего зоны ослабил полезную календарную часть текущей модели.

Числа округлены для чтения; критерии считались по полной точности.
Вариант А немного улучшил второе окно, а Б — два первых, но локальные
улучшения не превратились в снижение итогового S. Это результаты именно
зафиксированных конфигураций; они не доказывают бесполезность CatBoost вообще.

Сравнение пяти методов заняло 304.1 с в этом окружении. Кандидаты с CatBoost
добавляют ещё одну модель и зависимость без измеренного выигрыша; явное
линейное взаимодействие проще, но тоже не оправдало замену.

Проверка доступности данных пройдена. Ограничение оценки сохраняется:
все три окна уже участвовали в выборе, поэтому истинная ошибка скрытого
test неизвестна и независимой итоговой оценки здесь нет.